# Orchestrator

This notebook runs the datasource adapter to produce canonical data and submits it to the `CanonicalHandler` for storage.

To (re-)run an experiment, simply change `dataset_id` and `version` and re-run.

## 1. Inputs

In [2]:
dataset_id = "unsw_nb15"
version = "v1"

In [3]:
from context_based_anomalous_flow_detector.data_processing.adapters.unsw_nb15_adapter import (
    create_canonical_data,
)
from context_based_anomalous_flow_detector.data_processing.canonical_handler import (
    CanonicalHandler,
)

## 3. Run

Instantiate a `CanonicalHandler` and submit the adapter function. The handler executes it and saves the canonical data to
`data/<dataset_id>/<version>/canonical/canonical.parquet`.

In [6]:
handler = CanonicalHandler()
target_dir = handler.create(create_canonical_data, dataset_id, version, "NF-UNSW-NB15-v3.csv")

print("Canonical data written to:", target_dir)

alles in ordnung
Canonical data saved to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/canonical/canonical.parquet
Canonical data written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/canonical


## 4. Inspect result

In [7]:
import pandas as pd

canonical = pd.read_parquet(target_dir / "canonical.parquet")
canonical.head()

,bytes_out,bytes_in,packets_out,packets_in,tcpflags,ttl,ipsrc,ipdst,portsrc,portdst,duration,starttime,proto,label,attack
0,178,146,2,2,0,31,59.166.0.2,149.171.126.3,4894,53,2,1424242193040,17,0,Benign
1,2976,4704,28,28,27,32,59.166.0.4,149.171.126.6,52671,31992,335,1424242192744,6,0,Benign
2,548216,13662,438,238,27,32,59.166.0.0,149.171.126.9,47290,6881,2460,1424242190649,6,0,Benign
3,178,146,2,2,0,31,59.166.0.8,149.171.126.7,43310,53,1,1424242193145,17,0,Benign
4,162,130,2,2,0,31,59.166.0.1,149.171.126.1,45870,53,1,1424242193239,17,0,Benign


In [8]:
print(f"Rows: {len(canonical):,}")
print(f"Columns: {list(canonical.columns)}")

Rows: 2,365,424
Columns: ['bytes_out', 'bytes_in', 'packets_out', 'packets_in', 'tcpflags', 'ttl', 'ipsrc', 'ipdst', 'portsrc', 'portdst', 'duration', 'starttime', 'proto', 'label', 'attack']


## 5. Preprocessing

The `PreprocessingHandler` reads the canonical data, validates it against the schema, and produces model-input tensors.
It records provenance: which canonical data (dataset_id/version) was consumed and the preprocessing configuration.

In [9]:
from context_based_anomalous_flow_detector.data_processing.preprocessing import PreprocessParams
from context_based_anomalous_flow_detector.data_processing.preprocessing_handler import (
    PreprocessingHandler,
)

In [10]:
preprocessor = PreprocessingHandler()
params = PreprocessParams(seq_len=34, group_cols=["ipsrc"], time_col="starttime")
preprocessed_dir = preprocessor.run(dataset_id, version, params=params)

print("Preprocessed data written to:", preprocessed_dir)

Preprocessed tensors saved to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/preprocessed/tensors.pt
Preprocessed data written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/preprocessed


In [ ]:
import torch

tensors = torch.load(preprocessed_dir / "tensors.pt", weights_only=False)
for name, split in tensors.items():
    print(name, {k: tuple(v.shape) for k, v in split.items()})

## 6. Training

The `TrainerHandler` reads the preprocessed tensors of a dataset/version, trains an `AnoBert` model on the train split,
and saves the artifact plus a provenance manifest to `models/<dataset_id>/<version>/`.

In [ ]:
from context_based_anomalous_flow_detector.training.trainer_handler import (
    TrainerHandler,
    TrainingParams,
)

In [ ]:
trainer = TrainerHandler()
params = TrainingParams(
    seed=42, batch_size=64, num_epochs=50, d_model=128, n_heads=4, num_layers=4
)
model_dir = trainer.run(dataset_id, version, params=params)

print("Trained model written to:", model_dir)

In [ ]:
import yaml

from context_based_anomalous_flow_detector.data_processing._manifest import read_manifest

manifest = read_manifest(model_dir)
print(yaml.safe_dump(manifest, sort_keys=False))

### 6b. Resume training from a saved checkpoint

`load_trained_model` reconstructs an `AnoBert` (architecture + weights) from a saved run's `metadata.yaml` + `model.pt`,
so you can inspect an already-trained model without retraining.

To actually continue training it, call `TrainerHandler.run(..., resume=True)`: it warm-starts from the existing
checkpoint, continues the epoch numbering from where the previous run stopped, and merges the new metrics into the
existing manifest instead of overwriting the history.

In [ ]:
from context_based_anomalous_flow_detector.training.trainer_handler import load_trained_model

loaded_model, loaded_manifest = load_trained_model(dataset_id, version)
loaded_metrics = loaded_manifest["metrics"]

print(
    f"Loaded model trained for {loaded_metrics['stopped_epoch']} epochs; "
    f"weights are from epoch {loaded_metrics.get('best_epoch', '?')} "
    f"(best eval loss {loaded_metrics['best_eval_loss']:.4f})"
)

Continue training for more epochs. `resume=True` requires an existing checkpoint at `models/<dataset_id>/<version>/model.pt`.

In [ ]:
from dataclasses import replace

# Architecture fields must match the checkpoint being resumed (load_state_dict requires
# identical shapes), so derive resume_params from the loaded manifest rather than
# retyping them -- only override what's meant to change for the continuation.
resume_params = replace(
    TrainingParams.from_dict(loaded_manifest["training_params"]), num_epochs=20
)
model_dir = trainer.run(dataset_id, version, params=resume_params, resume=True)

print("Resumed training; model updated at:", model_dir)

In [ ]:
manifest = read_manifest(model_dir)
print(yaml.safe_dump(manifest, sort_keys=False))

## 7. Evaluation

Training uses masked-token prediction: ~15% of the real tokens are masked and the model must reconstruct them.
The `EvaluationHandler` applies the same masking to the held-out split and measures how well the model recovers
the masked tokens (continuous MSE/MAE, proto/port accuracy), alongside trivial baselines for context.

In [ ]:
from context_based_anomalous_flow_detector.evaluation.evaluation_handler import (
    EvalParams,
    EvaluationHandler,
)

In [ ]:
evaluator = EvaluationHandler()
eval_params = EvalParams(seed=42, mask_ratio=0.15, batch_size=64, top_k=5)
eval_model_dir = evaluator.run(dataset_id, version, split="test", params=eval_params)

print("Evaluation results written to:", eval_model_dir / "evaluation.json")

Inspect the raw evaluation results (metrics + baselines) from the saved JSON.

In [ ]:
import json

with open(eval_model_dir / "evaluation.json") as f:
    results = json.load(f)

print("metrics:", json.dumps(results["metrics"], indent=2))

## 8. Anomaly detection

The model was trained as a masked-token predictor: it only ever received a training signal for
positions that were masked, never for positions fed in unmasked. So scoring anomalies by feeding
a flow through unmasked and comparing directly to the output (a plain autoencoder approach) would
compare against never-trained, uncalibrated output.

`AnomalyDetectionHandler` instead mirrors the actual training task: for every real (non-padded)
flow, it masks exactly that flow's position and lets the model reconstruct it from its context --
leave-one-out over the whole sequence, so every flow gets a complete, deterministic score (unlike
the single random ~15% masking pass `EvaluationHandler` uses for aggregate metrics). It reports
both a single "combined" anomaly score and its raw, unweighted "cont"/"proto"/"port" components,
each checked against the ground-truth `label` column via ROC AUC / average precision.

The combine weights (`cont_weight`/`proto_weight`/`port_weight` on `AnomalyDetectionParams`) are
*not* the training loss weights (30/1/0.5) -- a per-component AUC breakdown originally showed
`port_err` was inversely correlated with attacks (many attacks reuse one target port, which the
model predicted well from context on the old raw 65536-way port vocab, i.e. camouflage) and that
`cont_weight=30` drowned out `proto`'s much stronger signal. The current defaults (1/1/0) raised
ROC AUC from 0.68 to ~0.97 on unsw_nb15/v1 test -- purely a scoring-time change, no retraining
needed.

Ports are now also quantized before training (top-`port_top_k` most frequent destination ports
each get their own class, everything else falls into one of 3 IANA range buckets -- see
`schema.port_vocab_size` / `preprocessing.build_port_vocab`), which shrank `port_dst_embedding` +
`reconstruction_port` from 92.6% of the model's parameters down to a small fraction, and flipped
`port_err`'s correlation to the expected direction. It's still too noisy to help the linear
combination (`port_weight=0` remains best), but it's no longer actively misleading.

In [ ]:
from context_based_anomalous_flow_detector.evaluation.anomaly_detection_handler import (
    AnomalyDetectionHandler,
    AnomalyDetectionParams,
)

In [ ]:
detector = AnomalyDetectionHandler()
ad_params = AnomalyDetectionParams(batch_size=64)
ad_model_dir = detector.run(dataset_id, version, split="test", params=ad_params)

print("Anomaly detection results written to:", ad_model_dir / "anomaly_detection.json")

In [ ]:
with open(ad_model_dir / "anomaly_detection.json") as f:
    ad_results = json.load(f)

for name, m in ad_results["metrics"].items():
    if "roc_auc" in m:
        print(f"{name:<9}: ROC AUC {m['roc_auc']:.4f}  AP {m['average_precision']:.4f}")

Plot the score distributions for benign vs. attack flows to see how well they separate.

In [ ]:
import matplotlib.pyplot as plt

scores_blob = torch.load(ad_model_dir / "anomaly_scores.pt", weights_only=True)
real = ~scores_blob["mask"]
labels = scores_blob["labels"][real].numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, key in zip(axes.flat, ("combined", "cont", "proto", "port")):
    values = scores_blob[key][real].numpy()
    ax.hist(values[labels == 0], bins=50, alpha=0.6, label="benign", density=True)
    ax.hist(values[labels == 1], bins=50, alpha=0.6, label="attack", density=True)
    ax.set_xlabel(f"{key} score")
    ax.set_ylabel("density")
    ax.legend()
    ax.set_title(key)

fig.suptitle(f"Anomaly score distributions ({dataset_id}/{version}, test split)")
fig.tight_layout()
plt.show()